# 03. Preparación de los datos

## 1. Objetivo
Limpiar, transformar y estructurar el dataset para dejarlo listo para la etapa de modelado, garantizando consistencia en las variables, tratamiento adecuado de valores inconsistentes, selección de variables útiles y generación de conjuntos de entrenamiento y prueba.

## 2. Carga de datos

In [75]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown
from src.data.make_dataset import (
    clean_column_names,
    clean_categorical_strings,
    fix_financial_stress
)

from config.paths import (
    RAW_DATA_FILE,
    CLEAN_DATA_FILE,
    X_TRAIN_FILE,
    X_TEST_FILE,
    Y_TRAIN_FILE,
    Y_TEST_FILE,
    ensure_project_dirs
)

pd.set_option("display.max_columns", None)
ensure_project_dirs()

df = pd.read_csv(RAW_DATA_FILE)
display(df.head())

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,'5-6 hours',Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,'5-6 hours',Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,'Less than 5 hours',Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,'7-8 hours',Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,'5-6 hours',Moderate,M.Tech,Yes,1.0,1.0,No,0


## 3. Limpieza de nombres de columnas, cadenas de texto categoricas (Quitar comillas simples), imputación de datos y conversión de datos en columna estres financiero

In [76]:
df = clean_column_names(df)
df = fix_financial_stress(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 18 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id                                    27901 non-null  int64  
 1   gender                                27901 non-null  object 
 2   age                                   27901 non-null  float64
 3   city                                  27901 non-null  object 
 4   profession                            27901 non-null  object 
 5   academic_pressure                     27901 non-null  float64
 6   work_pressure                         27901 non-null  float64
 7   cgpa                                  27901 non-null  float64
 8   study_satisfaction                    27901 non-null  float64
 9   job_satisfaction                      27901 non-null  float64
 10  sleep_duration                        27901 non-null  object 
 11  dietary_habits 

## 6. Estandarización de variables categóricas

In [77]:
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

for col in df.select_dtypes(include="object").columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- gender ---
gender
Male      15547
Female    12354
Name: count, dtype: int64

--- city ---
city
Kalyan           1570
Srinagar         1372
Hyderabad        1340
Vasai-Virar      1290
Lucknow          1155
Thane            1139
Ludhiana         1111
Agra             1094
Surat            1078
Kolkata          1066
Jaipur           1036
Patna            1007
Visakhapatnam     969
Pune              968
Ahmedabad         951
Name: count, dtype: int64

--- profession ---
profession
Student                     27870
Architect                       8
Teacher                         6
'Digital Marketer'              3
'Content Writer'                2
Chef                            2
Doctor                          2
Pharmacist                      2
'Civil Engineer'                1
'UX/UI Designer'                1
'Educational Consultant'        1
Manager                         1
Lawyer                          1
Entrepreneur                    1
Name: count, dtype: int64

--- sleep_

## 7. Revisión de variables con baja variabilidad

In [78]:
cols_baja_variabilidad = ["work_pressure", "job_satisfaction", "profession"]
df = df.drop(columns=cols_baja_variabilidad)

display(Markdown("## Variables eliminadas por baja variabilidad"))
display(pd.DataFrame({"variables_eliminadas": cols_baja_variabilidad}))

## Variables eliminadas por baja variabilidad

,variables_eliminadas
0,work_pressure
1,job_satisfaction
2,profession


Se eliminaron las variables work_pressure, job_satisfaction y profession debido a su muy baja variabilidad. En los tres casos, una sola categoría o valor concentra prácticamente la totalidad de los registros, por lo que su aporte discriminante al modelado sería muy limitado.

In [79]:
frecuencia_degree = df["degree"].value_counts()
categorias_frecuentes = frecuencia_degree[frecuencia_degree >= 500].index

df["degree_grouped"] = np.where(df["degree"].isin(categorias_frecuentes), df["degree"], "Other")
df = df.drop(columns=["degree"])

display(Markdown("## Distribución de degree_grouped"))
display(df["degree_grouped"].value_counts().to_frame("frecuencia"))

## Distribución de degree_grouped

,frecuencia
degree_grouped,
'Class 12',6080
B.Ed,1867
B.Com,1506
B.Arch,1478
BCA,1433
MSc,1190
B.Tech,1152
MCA,1044
M.Tech,1022


In [80]:
frecuencia_city = df["city"].value_counts()
ciudades_frecuentes = frecuencia_city[frecuencia_city >= 500].index

df["city_grouped"] = np.where(df["city"].isin(ciudades_frecuentes), df["city"], "Other")
df = df.drop(columns=["city"])

display(Markdown("## Distribución de city_grouped"))
display(df["city_grouped"].value_counts().to_frame("frecuencia").head(15))

## Distribución de city_grouped

,frecuencia
city_grouped,
Kalyan,1570
Srinagar,1372
Hyderabad,1340
Vasai-Virar,1290
Lucknow,1155
Thane,1139
Ludhiana,1111
Agra,1094
Surat,1078


Aunque se identificaron algunos valores atípicos en variables como age, cgpa y job_satisfaction, su proporción es mínima respecto al total de registros, por lo que en esta etapa no se realizará eliminación de observaciones. Se prioriza conservar la información y evaluar posteriormente si estos valores afectan el desempeño de los modelos.

## 8. Selección de variables finales

In [81]:
df_model = df.copy()

display(Markdown("## Vista del dataset final para modelado"))
display(df_model.head())

display(Markdown("## Dimensiones del dataset final"))
display(pd.DataFrame({
    "filas": [df_model.shape[0]],
    "columnas": [df_model.shape[1]]
}))
df_model.to_csv(CLEAN_DATA_FILE, index=False)

## Vista del dataset final para modelado

,id,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,have_you_ever_had_suicidal_thoughts_,work_study_hours,financial_stress,family_history_of_mental_illness,depression,degree_grouped,city_grouped
0,2,Male,33.0,5.0,8.97,2.0,'5-6 hours',Healthy,Yes,3.0,1.0,No,1,B.Pharm,Visakhapatnam
1,8,Female,24.0,2.0,5.90,5.0,'5-6 hours',Moderate,No,3.0,2.0,Yes,0,BSc,Bangalore
2,26,Male,31.0,3.0,7.03,5.0,'Less than 5 hours',Healthy,No,9.0,1.0,Yes,0,BA,Srinagar
3,30,Female,28.0,3.0,5.59,2.0,'7-8 hours',Moderate,Yes,4.0,5.0,Yes,1,BCA,Varanasi
4,32,Female,25.0,4.0,8.13,3.0,'5-6 hours',Moderate,Yes,1.0,1.0,No,0,M.Tech,Jaipur


## Dimensiones del dataset final

,filas,columnas
0,27901,15


## 9. Separación X e Y (Variables predictoras y Objetivo)

In [82]:
X = df_model.drop(columns=["depression", "id"], errors="ignore")
y = df_model["depression"]

display(Markdown("## Dimensiones de X e y"))
display(pd.DataFrame({
    "conjunto": ["X", "y"],
    "filas": [X.shape[0], y.shape[0]],
    "columnas": [X.shape[1], 1]
}))

## Dimensiones de X e y

,conjunto,filas,columnas
0,X,27901,13
1,y,27901,1


## 10. División train/test

In [83]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(Markdown("## Dimensiones de train y test"))
display(pd.DataFrame({
    "dataset": ["X_train", "X_test", "y_train", "y_test"],
    "filas": [X_train.shape[0], X_test.shape[0], y_train.shape[0], y_test.shape[0]],
    "columnas": [X_train.shape[1], X_test.shape[1], 1, 1]
}))

## Dimensiones de train y test

,dataset,filas,columnas
0,X_train,22320,13
1,X_test,5581,13
2,y_train,22320,1
3,y_test,5581,1


## 11. Exportación de datasets procesados

In [84]:
from config.paths import CLEAN_DATA_FILE

df_model.to_csv(CLEAN_DATA_FILE, index=False)

X_train.to_csv(X_TRAIN_FILE, index=False)
X_test.to_csv(X_TEST_FILE, index=False)
y_train.to_csv(Y_TRAIN_FILE, index=False)
y_test.to_csv(Y_TEST_FILE, index=False)

archivos_generados = pd.DataFrame({
    "archivo": [
        "students_depression_clean.csv",
        "X_train.csv",
        "X_test.csv",
        "y_train.csv",
        "y_test.csv"
    ],
    "ruta": [
        str(CLEAN_DATA_FILE),
        str(X_TRAIN_FILE),
        str(X_TEST_FILE),
        str(Y_TRAIN_FILE),
        str(Y_TEST_FILE)
    ]
})

display(Markdown("## Archivos generados"))
display(archivos_generados)

## Archivos generados

,archivo,ruta
0,students_depression_clean.csv,C:\Users\HOME\Documents\Enfasis Aprendizaje de...
1,X_train.csv,C:\Users\HOME\Documents\Enfasis Aprendizaje de...
2,X_test.csv,C:\Users\HOME\Documents\Enfasis Aprendizaje de...
3,y_train.csv,C:\Users\HOME\Documents\Enfasis Aprendizaje de...
4,y_test.csv,C:\Users\HOME\Documents\Enfasis Aprendizaje de...


## 12. Conclusiones

En esta fase se realizaron las transformaciones necesarias para dejar el dataset listo para modelado. Se limpiaron los nombres de las columnas, se corrigieron valores inconsistentes en financial_stress, se eliminaron comillas y problemas de formato en variables categóricas y se ajustaron tipos de datos.

Adicionalmente, se eliminaron variables con muy baja variabilidad, como work_pressure, job_satisfaction y profession, debido a su escaso aporte potencial al poder explicativo del modelo. También se agruparon categorías poco frecuentes en variables de alta cardinalidad, como degree y city, con el fin de reducir ruido y mejorar la robustez del modelado.

Finalmente, se definieron las variables predictoras y la variable objetivo, y se realizó la partición en conjuntos de entrenamiento y prueba de forma estratificada. Con ello, la base quedó preparada para la fase de modelado dentro de la metodología CRISP-DM.